# Treated MonoCulture Math Modelling — 20k IC50 (Julia)

Uses DifferentialEquations.jl + Optimization.jl with an ODEProblem and a Hill-type drug effect, mirroring the style of the untreated notebook.

In [2]:
using Pkg
# Toggle installs if you hit missing packages
if false
    Pkg.activate(temp=false)
    Pkg.add([
        "CSV", "DataFrames", "Statistics", "DifferentialEquations",
        "Optimization", "OptimizationOptimJL", "BlackBoxOptim", "Plots", "Optim"
    ])
end
using CSV, DataFrames, Statistics
using DifferentialEquations, Optimization, OptimizationOptimJL, BlackBoxOptim
# Ensure Optim (needed for NelderMead) is available
try
    using Optim
catch e
    @warn "Optim.jl not found, attempting to install..." e
    Pkg.add("Optim")
    using Optim
end
using Plots
default(fmt=:png, legend=:topright, lw=2, size=(900,550))

ROOT = raw"c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics"
UNT_PARAMS = joinpath(ROOT, "Modelling Data Notebooks", "Untreated MonoCulture", "untreated_logistic_params_20k.csv")
TREATED_DIR = joinpath(ROOT, "Processed_Datasets", "Treated MonoCulture", "20k", "IC50", "Averages")
OUT_DIR = joinpath(ROOT, "Modelling Data Notebooks", "Treated MonoCulture", "20k", "IC50")
mkpath(OUT_DIR)

function load_day_averages(path::AbstractString)
    df = CSV.read(path, DataFrame)
    daycol = :Day ∈ names(df) ? :Day : Symbol(first(filter(n->occursin("day", lowercase(String(n))), names(df))))
    valcol = Symbol("Mean Cells") ∈ names(df) ? Symbol("Mean Cells") : Symbol(first(filter(n->occursin("mean", lowercase(String(n))) && occursin("cells", lowercase(String(n))), names(df))))
    x = Float64.(df[!, daycol]); y = Float64.(df[!, valcol])
    perm = sortperm(x); x=x[perm]; y=y[perm]
    return x, y
end

function load_untreated_rK()
    df = CSV.read(UNT_PARAMS, DataFrame)
    return Dict(Symbol(r.cell_line)=> (r.r, r.K) for r in eachrow(df))
end

# Logistic growth with Hill-type drug effect as reduction in effective growth rate
# du/dt = (r * (1 - H(dose; IC50, n)))*u*(1 - u/K)
hill(dose, IC50, n) = 1.0 / (1.0 + (IC50 / max(dose, eps()))^n)
function treated_logistic!(du, u, p, t)
    r, K, IC50, n, dose = p
    eff = r * (1 - hill(dose, IC50, n))
    du[1] = eff * u[1] * (1 - u[1]/K)
end

# Build and solve with given parameters at sample times x
function solve_model(x::Vector{Float64}, y::Vector{Float64}, r, K; IC50=1.0, n=1.0, dose=1.0)
    u0 = [max(y[1], eps())]; tspan=(x[1], x[end])
    p = [r, K, IC50, n, dose]
    prob = ODEProblem(treated_logistic!, u0, tspan, p)
    sol = solve(prob, Rosenbrock23(); saveat=x, reltol=1e-9, abstol=1e-9)
    return sol
end

# Objective for Optimization.jl (squared error)
function objective(θ, x, y, r, K, dose)
    IC50, n = θ
    sol = solve_model(x, y, r, K; IC50=IC50, n=n, dose=dose)
    pred = getindex.(sol.u, 1)
    return sum(abs2, y .- pred)
end

# Helper to fit IC50 and n for a given cell line and CSV
function fit_ic50_for_csv(csv_path::AbstractString, rK::Tuple{Float64,Float64}; dose=1.0)
    x, y = load_day_averages(csv_path)
    r, K = rK
    θ0 = [1.0, 1.0]
    lower = [1e-3, 0.1]; upper=[1e3, 5.0]
    loss(θ) = objective(θ, x, y, r, K, dose)
    optf = OptimizationFunction((θ, p)->loss(θ), Optimization.AutoForwardDiff())
    prob = Optimization.OptimizationProblem(optf, θ0)
    # Qualify NelderMead with Optim namespace to avoid dispatch issues
    res = Optimization.solve(prob, Optim.Fminbox(Optim.NelderMead()), 
                         lower=lower, upper=upper, maxiters=2000)
    θ̂ = res.u
    IC50̂, n̂ = θ̂
    sol̂ = solve_model(x, y, r, K; IC50=IC50̂, n=n̂, dose=dose)
    ssr = objective(θ̂, x, y, r, K, dose)
    return (; x, y, r, K, IC50=IC50̂, n=n̂, ssr, sol=sol̂)
end

# Wire up actual 20k/IC50 treated averages CSVs
naive_csv = joinpath(TREATED_DIR, "A2780Naive_day_averages.csv")
cis_csv   = joinpath(TREATED_DIR, "A2780cis_day_averages.csv")

println("Looking for:\n  " * naive_csv * "\n  " * cis_csv)

rk = load_untreated_rK()
@assert haskey(rk, :A2780Naive) && haskey(rk, :A2780cis) "Missing untreated r,K export for 20k"

fits = Dict{Symbol,Any}()
if isfile(naive_csv)
    fits[:A2780Naive] = fit_ic50_for_csv(naive_csv, rk[:A2780Naive]; dose=1.0)
else
    @warn "Naive treated 20k IC50 CSV not found" naive_csv
end
if isfile(cis_csv)
    fits[:A2780cis] = fit_ic50_for_csv(cis_csv, rk[:A2780cis]; dose=1.0)
else
    @warn "Cis treated 20k IC50 CSV not found" cis_csv
end

for (k, f) in fits
    plt = scatter(f.x, f.y; label=String(k)*" data", xlabel="Day", ylabel="Cells", title=String(k)*" — 20k IC50 fit")
    plot!(plt, f.sol.t, getindex.(f.sol.u,1); label="Model")
    display(plt)
end

out_csv = joinpath(OUT_DIR, "treated_ic50_20k_fit_params.csv")
rows = DataFrame(cell_line=String[], r=Float64[], K=Float64[], IC50=Float64[], n=Float64[], SSR=Float64[])
for name in (:A2780Naive, :A2780cis)
    if haskey(fits, name)
        f = fits[name]
        push!(rows, (String(name), f.r, f.K, f.IC50, f.n, f.ssr))
    end
end
CSV.write(out_csv, rows)
println("Saved params → " * out_csv)

┌ Warning: Optim.jl not found, attempting to install...
│   e =
│    ArgumentError: Package Optim not found in current path, maybe you meant `import/using .Optim`.
│    - Otherwise, run `import Pkg; Pkg.add("Optim")` to install the Optim package.
└ @ Main In[2]:16
    Updating registry at `C:\Users\MainFrameTower\.julia\registries\General.toml`
    Updating registry at `C:\Users\MainFrameTower\.julia\registries\General.toml`
   Resolving package versions...
    Updating `C:\Users\MainFrameTower\.julia\environments\v1.11\Project.toml`
  [429524aa] + Optim v1.13.2
  No Changes to `C:\Users\MainFrameTower\.julia\environments\v1.11\Manifest.toml`


Looking for:
  c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics\Processed_Datasets\Treated MonoCulture\20k\IC50\Averages\A2780Naive_day_averages.csv
  c:/Users/MainFrameTower/Desktop/CancerGrowthDynamics\Processed_Datasets\Treated MonoCulture\20k\IC50\Averages\A2780cis_day_averages.csv


LoadError: MethodError: no method matching Optim.Options(; extended_trace::Bool, lb::Vector{Float64}, ub::Vector{Float64}, callback::OptimizationOptimJL.var"#_cb#12"{OptimizationCache{OptimizationFunction{true, AutoForwardDiff{nothing, Nothing}, var"#20#22"{var"#loss#21"{Float64, Float64, Float64, Vector{Float64}, Vector{Float64}}}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, typeof(SciMLBase.DEFAULT_OBSERVED_NO_TIME), Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing}, OptimizationBase.ReInitCache{Vector{Float64}, SciMLBase.NullParameters}, Nothing, Nothing, Nothing, Nothing, Nothing, NelderMead{Optim.AffineSimplexer, Optim.AdaptiveParameters}, Bool, OptimizationOptimJL.var"#4#6", Nothing}}, iterations::Int64)
This error has been manually thrown, explicitly, so the method may exist but be intentionally marked as unimplemented.

[0mClosest candidates are:
[0m  Optim.Options

In [ ]:
# Quick check: use discovered filenames
TREATED_DIR = joinpath(ROOT, "Processed_Datasets", "Treated MonoCulture", "20k", "IC50", "Averages")
naive_csv = joinpath(TREATED_DIR, "A2780Naive_day_averages.csv")
cis_csv   = joinpath(TREATED_DIR, "A2780cis_day_averages.csv")
println("Using:\n  " * naive_csv * "\n  " * cis_csv)